# Predict Seagrass, Ulva and Gracilaria
* Across Sentinel-2 date range
* Across council priority areas not used in training (execpt for WRC - this was used in training)
* Using the selected U-Net

In [ ]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling
import datetime

%load_ext autoreload
%autoreload 2

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

# Read in sites and standardise

In [ ]:
data_path = utils.get_data_path()
council_site_path = data_path / "site_polygons" / "council_sites" / "standardised"

council_sites = {"AshleyRiver_Estuary.zip/AshleyRiver_Estuary": "Ashley Mouth ECAN",
                 "Brookland_Lagoon.zip/Brookland Lagoon": "Brookland Lagoon ECAN",
                 "porirua_intertidal_extent_GW.zip": "Porirua Estuary GWRC",
                 "Nelson Haven from OSM.geojson": "Nelson Haven NCC",
                 "Manukau_estuary_polygon_MHWS.zip/Manukau_estuary_polygon_MHWS": "Manukau Estuary AC"}

for key, value in council_sites.items():
    if not (council_site_path / f"{value}.gpkg").exists():
        print(f"Save {value}")
        if "zip" in key:
            zip_path = council_site_path.parent / key
            boundary = geopandas.read_file(f"/vsizip/{zip_path.as_posix()}")
        else:
            boundary = geopandas.read_file(council_site_path.parent / key)
        boundary.to_crs(utils.CRS_NZTM).to_file(council_site_path / f"{value}.gpkg")
    else:
        print(f"Already created {value}")

# Value to edit

In [ ]:
prediction_sites =  list(council_sites.values())

In [ ]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .5
model_max_cloud_cover = 5 # percentage
model_low_tide_delta_hrs = 1
model_low_tide_delta_mins = 0
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage
predict_low_tide_delta_hrs = 0
predict_low_tide_delta_mins = 30

num_epoch=150
tile_size=128
tile_stride=64

filtered=True
filtered_label = "filtered_" if filtered else ""

model_description = f"UN_model_{filtered_label}dates_13_class_{int(test_threshold*100)}_percent_test_{num_epoch}_epochs_{tile_size}_tiling_{tile_stride}_stride"  
unet_type = "diffuser_unet"
model_name = "diffuser-unet-epoch=98-val_loss=0.5173.ckpt"

website_tab = (
    f"{model_description}_train_{utils.get_samples_folder(sample_method=sample_method, method_2_threshold=method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins, max_cloud_cover=model_max_cloud_cover)}_"
    f"predict_{utils.get_low_tide_max_cloud_name(low_tide_delta_hrs=predict_low_tide_delta_hrs, low_tide_delta_mins=predict_low_tide_delta_mins, max_cloud_cover=predict_max_cloud_cover)}"
)

In [ ]:
model_path = utils.get_models_path(sample_method=sample_method,
                                   method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs,
                                   low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_path = model_path / model_description / unet_type
model_file = model_path / model_name
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(
    sample_method=sample_method, method_2_threshold=method_2_threshold,
    model_max_cloud_cover=model_max_cloud_cover,
    predict_max_cloud_cover=predict_max_cloud_cover,
    predict_low_tide_delta_hrs=predict_low_tide_delta_hrs,
    predict_low_tide_delta_mins=predict_low_tide_delta_mins,
    model_low_tide_delta_hrs=model_low_tide_delta_hrs,
    model_low_tide_delta_mins=model_low_tide_delta_mins,
    filtered=filtered, use_council_paths=True
)
predictions_path = predictions_path / model_description / unet_type
predictions_path.mkdir(parents=True, exist_ok=True)

website_root = data_path / "website" / "council_sites" / website_tab
website_root.mkdir(exist_ok=True, parents=True)

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

### Predict values

In [ ]:
for prediction_site in prediction_sites[:4]:

    website_output_folder = website_root / prediction_site
    website_output_folder.mkdir(exist_ok=True, parents=True)
    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already predicted and prepared for website. Skipping. Delete if unexpected.")
        continue

    
    polygon_file = council_site_path / f"{prediction_site}.gpkg"
    site_polygon = geopandas.read_file(polygon_file)

    ## Predict
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            #model_feature_names_file = model_file.with_name(f"{model_file.stem}_feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site_unet(
                test_satellite_file=satellite_file,
                polygon_file=polygon_file,
                checkpoint_file=model_file,
                weights_only=False,
                tile_size=tile_size,
                stride=tile_stride)
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name(f"class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name, "satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = prediction.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)

    ## Website
    website_options = {"value": prediction_sites, "label": prediction_sites}
    website_options = pandas.DataFrame(website_options)
    website_options.to_csv(website_root / "options.csv", index=False)

    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_{target.lower()}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'download_satellite_predictions.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(website_output_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["B04_scale"].append(precentage_dict["B04"])
                    info["B03_scale"].append(precentage_dict["B03"])
                    info["B02_scale"].append(precentage_dict["B02"]) 
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs",
                                "B04_scale": "B04 Rescale",
                                "B03_scale": "B03 Rescale",
                                "B02_scale": "B02 Rescale"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")

        if not value:
            raise ValueError(f"No {key} prediction files for {prediction_site}")
    
        target_extents = pandas.concat(value, ignore_index=True)
        target_dates = pandas.to_datetime(target_extents["date"], utc=True)
        info_dates = pandas.to_datetime(info["date"], utc=True)
    
        if target_dates.isna().any() or info_dates.isna().any():
            raise ValueError(f"Missing date for {prediction_site}, {key}")
    
        if target_dates.duplicated().any():
            raise ValueError(f"Duplicate {key} dates for {prediction_site}")
    
        missing_metadata = set(target_dates) - set(info_dates)
        if missing_metadata:
            raise ValueError(f"Missing satellite metadata for {key}: {missing_metadata}")
    
        target_extents["area"] = target_extents.geometry.area
        area_by_date = pandas.Series(
            target_extents["area"].to_numpy(),
            index=target_dates,
        )
        info[f"{key} Area [m^2]"] = info_dates.map(area_by_date)
    
        target_extents.dissolve()[["geometry"]].to_file(
            website_output_folder / f"{key.lower()}_presence_absence_map.gpkg"
        )

    
    info.to_csv(website_output_folder / "info_all_dates.csv", index=False)
    

                

# 10% unfiltered cloud threshold for Manakau
* Note that the model is filtered
* The prediction values are not
* Use both 64 and 32 stride to see if this addresses the grid artefact noticable in the 64 stride predictions

In [ ]:
predict_stride = 32

In [ ]:
predict_stride_string = "" if predict_stride == tile_stride else f"_inference_stride_{predict_stride}"

predict_max_cloud_cover = 10 # percentage"

website_tab = (
    f"{model_description}_train_{utils.get_samples_folder(sample_method=sample_method, method_2_threshold=method_2_threshold)}_"
    f"{utils.get_low_tide_max_cloud_name(low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins, max_cloud_cover=model_max_cloud_cover)}_"
    f"predict_{utils.get_low_tide_max_cloud_name(low_tide_delta_hrs=predict_low_tide_delta_hrs, low_tide_delta_mins=predict_low_tide_delta_mins, max_cloud_cover=predict_max_cloud_cover)}{predict_stride_string}"
)

data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(
    sample_method=sample_method, method_2_threshold=method_2_threshold,
    model_max_cloud_cover=model_max_cloud_cover,
    predict_max_cloud_cover=predict_max_cloud_cover,
    predict_low_tide_delta_hrs=predict_low_tide_delta_hrs,
    predict_low_tide_delta_mins=predict_low_tide_delta_mins,
    model_low_tide_delta_hrs=model_low_tide_delta_hrs,
    model_low_tide_delta_mins=model_low_tide_delta_mins,
    filtered=False, use_council_paths=True
)
predictions_path = predictions_path / model_description / f"{unet_type}{predict_stride_string}"
predictions_path.mkdir(parents=True, exist_ok=True)

website_root = data_path / "website" / "council_sites" / website_tab
website_root.mkdir(exist_ok=True, parents=True)

In [ ]:
for prediction_site in prediction_sites[-1:]:

    website_output_folder = website_root / prediction_site
    website_output_folder.mkdir(exist_ok=True, parents=True)
    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already predicted and prepared for website. Skipping. Delete if unexpected.")
        continue

    
    polygon_file = council_site_path / f"{prediction_site}.gpkg"
    site_polygon = geopandas.read_file(polygon_file)

    ## Predict
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}.nc"
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            #model_feature_names_file = model_file.with_name(f"{model_file.stem}_feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site_unet(
                test_satellite_file=satellite_file,
                polygon_file=polygon_file,
                checkpoint_file=model_file,
                weights_only=False,
                tile_size=tile_size,
                stride=predict_stride)
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name(f"class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name, "satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = prediction.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)

    ## Website
    website_options = {"value": prediction_sites, "label": prediction_sites}
    website_options = pandas.DataFrame(website_options)
    website_options.to_csv(website_root / "options.csv", index=False)

    if (website_output_folder / "info_all_dates.csv").exists():
        print(f"{prediction_site} already run. Go to next")
        continue
    extents_all_years = {"Seagrass": [], "Ulva": [], "Gracilaria": []}
    info = {"ids": [], "B04_scale":[], "B03_scale":[], "B02_scale": [], "date":[]}
    for year in years:
        print(f"{prediction_site}: year {year}")
        satellite_info_set = False
        for target in extents_all_years.keys():
            target_file = predictions_path / f"{prediction_site}_{year}_{target.lower()}.gpkg"

            if not target_file.exists():
                print(f"WARNING no {target_file.name} file. Run 'download_satellite_predictions.ipynb' is this is unexpected.")
                continue
            target_extents = geopandas.read_file(target_file)
            extents_all_years[target].append(target_extents)

            for index, row in target_extents.iterrows():
                target_extent = geopandas.GeoDataFrame(geometry=[row["geometry"]], crs=utils.CRS_NZTM)
                date_YYMMDD = datetime.datetime.strptime(row.date, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%d')
        
                filename = f"{date_YYMMDD}_{target.lower()}.gpkg"
                target_extent.to_file(website_output_folder / filename)

                # Look up the satellite tile information
                if not satellite_info_set:
                    #print(f"\tGet Tile(s) ID: {date_YYMMDD}")
                    tile_id, precentage_dict = sentinel2.get_satellite_info(geometry=site_polygon, date_YYMMDD=date_YYMMDD)
                    
                    info["date"].append(row.date)
                    info["ids"].append(tile_id)
                    info["B04_scale"].append(precentage_dict["B04"])
                    info["B03_scale"].append(precentage_dict["B03"])
                    info["B02_scale"].append(precentage_dict["B02"]) 
            satellite_info_set = True
            
    info = pandas.DataFrame(info)
    info = info.rename(columns={"ids": "Satellite Tile IDs",
                                "B04_scale": "B04 Rescale",
                                "B03_scale": "B03 Rescale",
                                "B02_scale": "B02 Rescale"})
    
    for key, value in extents_all_years.items():
        print(f"\tTarget {key}")

        if not value:
            raise ValueError(f"No {key} prediction files for {prediction_site}")
    
        target_extents = pandas.concat(value, ignore_index=True)
        target_dates = pandas.to_datetime(target_extents["date"], utc=True)
        info_dates = pandas.to_datetime(info["date"], utc=True)
    
        if target_dates.isna().any() or info_dates.isna().any():
            raise ValueError(f"Missing date for {prediction_site}, {key}")
    
        if target_dates.duplicated().any():
            raise ValueError(f"Duplicate {key} dates for {prediction_site}")
    
        missing_metadata = set(target_dates) - set(info_dates)
        if missing_metadata:
            raise ValueError(f"Missing satellite metadata for {key}: {missing_metadata}")
    
        target_extents["area"] = target_extents.geometry.area
        area_by_date = pandas.Series(
            target_extents["area"].to_numpy(),
            index=target_dates,
        )
        info[f"{key} Area [m^2]"] = info_dates.map(area_by_date)
    
        target_extents.dissolve()[["geometry"]].to_file(
            website_output_folder / f"{key.lower()}_presence_absence_map.gpkg"
        )

    
    info.to_csv(website_output_folder / "info_all_dates.csv", index=False)
    

                